# Customer Churn Prediction
## Notebook 4 of 8 — Feature Engineering

This project predicts which telecom customers are likely to **churn** (cancel their service) so the business can reach them with retention offers *before* they leave. It walks through the full data-science lifecycle — exploration, cleaning, EDA, feature engineering, modelling, evaluation, and a tuned final model.

**Business problem:** Winning a new customer costs far more than keeping an existing one. This telecom loses roughly **27% of its customers**, and the leadership team wants a reliable, data-driven way to flag at-risk customers early enough to act.

**Tools & techniques:** Python · pandas · NumPy · Matplotlib · Seaborn · scikit-learn · XGBoost · SMOTE (imbalanced-learn) · joblib

> **This notebook:** we drop redundant columns and encode every categorical feature into numbers, then save the model-ready matrix that all modelling notebooks load.

**Author:** La Yaung Linn Lett  &nbsp;·&nbsp;  **Last updated:** June 2026

---

## 1. Load the cleaned data

**What:** Read the cleaned dataset from Notebook 02.

**Why:** We start from the single cleaned source so encoding is the only transformation happening here.

In [1]:
# Standard library
import warnings

# Third-party
import pandas as pd
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=UserWarning)

df = pd.read_csv("../data/processed/telco_churn_clean.csv")
print(f"Loaded cleaned data: {df.shape}")

C:\Users\Vivobook\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loaded cleaned data: (7043, 21)


## 2. Drop columns that don't help the model

**What:** Remove `customerID` and `TotalCharges`.

**Why:**
- `customerID` is a unique identifier — pure noise with no predictive value.
- `TotalCharges` is highly correlated with `tenure` × `MonthlyCharges` (seen in the EDA heatmap), so dropping it removes redundancy and reduces multicollinearity.

In [2]:
df = df.drop(columns=["customerID", "TotalCharges"])
print(f"Shape after dropping columns: {df.shape}")

Shape after dropping columns: (7043, 19)


Down to **19 columns**. Every remaining column is a genuine candidate predictor of churn.

## 3. Encode binary (yes/no) columns

**What:** Label-encode the columns that have exactly two categories into 0/1.

**Why:** Models need numbers, not text. For two-category columns a simple 0/1 mapping is the most compact, lossless encoding.

In [3]:
# Columns with exactly two categories -> map to 0/1
binary_cols = ["gender", "Partner", "Dependents",
               "PhoneService", "PaperlessBilling", "Churn"]

label_encoder = LabelEncoder()
for col in binary_cols:
    df[col] = label_encoder.fit_transform(df[col])

df[binary_cols].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling,Churn
0,0,1,0,0,1,0
1,1,0,0,1,0,0
2,1,0,0,1,1,1
3,1,0,0,0,0,0
4,0,0,0,1,1,1


The binary columns — including the **`Churn` target** (No -> 0, Yes -> 1) — are now numeric 0/1 values.

## 4. One-hot encode multi-category columns

**What:** Convert the remaining text columns (those with 3+ categories, e.g. `Contract`, `PaymentMethod`, `InternetService`) into one-hot dummy columns.

**Why:** These categories have no natural order, so one-hot encoding avoids implying a false ranking. `drop_first=True` removes one redundant column per feature to prevent the dummy-variable trap.

In [4]:
df = pd.get_dummies(df, drop_first=True)
print(f"Shape after one-hot encoding: {df.shape}")

Shape after one-hot encoding: (7043, 30)


Encoding expands the table to **30 numeric columns** (29 features + the `Churn` target). Everything is now numeric and model-ready.

## 5. Save the model-ready matrix

**What:** Write the fully encoded data to `data/processed/model_ready.csv`.

**Why:** Every modelling notebook (05-08) loads this one file, so the exact same feature set is used everywhere — no silent inconsistencies between experiments. (The clearer name also fixes the old, misleading `cleaned_data.csv`, which was actually the encoded matrix.)

In [5]:
df.to_csv("../data/processed/model_ready.csv", index=False)
print("Saved -> data/processed/model_ready.csv")
print(f"Final shape: {df.shape}")

Saved -> data/processed/model_ready.csv
Final shape: (7043, 30)


## Section conclusion — features are model-ready

- Dropped **`customerID`** (no signal) and **`TotalCharges`** (redundant / collinear).
- **Label-encoded** 6 binary columns into 0/1.
- **One-hot encoded** the multi-category columns (with `drop_first` to avoid the dummy trap).
- Produced a clean **7,043 × 30** numeric matrix and saved it as the single input for all modelling.

**Next:** Notebook 05 builds a baseline model on this matrix.